# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [3]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [9]:
context = [ {'role':'system', 'content':"""
 CREATE SEVERAL (3+) TABLES HERE
"""} ]



In [10]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
WRITE IN YOUR CONTEXT QUERIES HERE
"""
})

In [12]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [18]:
#new
context_user = context.copy()
print(return_CCRMSQL("The name of the employee best paid" , context_user))

```sql
SELECT name
FROM employees
ORDER BY salary DESC
LIMIT 1;
```


In [19]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("The name of the employee best paid", old_context_user))

This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query retrieves the name of the employee who is best paid by joining the "employees" table with the "salary" table on the ID_usr field. It then orders the result by salary in descending order and limits the output to only one row, which corresponds to the employee with the highest salary.


In [21]:
#new
print(return_CCRMSQL("Return the Institution with a higher average salary", context_user))

```sql
SELECT Institution, AVG(Salary) AS AverageSalary
FROM Salaries
GROUP BY Institution
ORDER BY AverageSalary DESC
LIMIT 1;
```


In [22]:
#old
print(return_CCRMSQL("Return the Institution with a higher average salary", old_context_user))

This is your SQL:
```sql
SELECT s.Institution, AVG(s.salary) AS avg_salary
FROM salary s
JOIN studies st ON s.ID_usr = st.ID_usr
GROUP BY s.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```

This SQL query joins the `salary` and `studies` tables on the `ID_usr` column, calculates the average salary for each institution, and then selects the institution with the highest average salary.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

# SQL 3 Versions Prompt Comparison - Lab Exercise Runner

1.1 Importing corresponding libraries

In [23]:
from copy import deepcopy
import csv, os

2.2 Defining prompt variants

In [29]:
# -------- Variant A: Minimal tables & columns (baseline) --------
context_A = [
  {'role':'system','content': "You are a Text-to-SQL assistant. Output only the final SQL for SQLite."},
  {'role':'system','content': """
Tables:
  customers(customer_id, name, email, city)
  products(product_id, name, category, price)
  orders(order_id, customer_id, order_date, status)
  order_items(order_id, product_id, quantity, unit_price)
Foreign keys:
  orders.customer_id -> customers.customer_id
  order_items.order_id -> orders.order_id
  order_items.product_id -> products.product_id
"""}
]

# -------- Variant B: DDL with constraints (no sample rows) --------
context_B = [
  {'role':'system','content': """You are a Text-to-SQL assistant. 
- Output only SQL (SQLite). 
- Use explicit JOIN ... ON ...
- Do not invent columns or tables.
- If impossible, answer exactly: 'I cannot answer that with the given schema.'"""},
  {'role':'system','content': """
CREATE TABLE customers (
  customer_id INTEGER PRIMARY KEY,
  name TEXT NOT NULL,
  email TEXT UNIQUE,
  city TEXT
);
CREATE TABLE products (
  product_id INTEGER PRIMARY KEY,
  name TEXT NOT NULL,
  category TEXT,
  price REAL NOT NULL
);
CREATE TABLE orders (
  order_id INTEGER PRIMARY KEY,
  customer_id INTEGER NOT NULL,
  order_date TEXT NOT NULL, -- YYYY-MM-DD
  status TEXT,
  FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
CREATE TABLE order_items (
  order_id INTEGER NOT NULL,
  product_id INTEGER NOT NULL,
  quantity INTEGER NOT NULL,
  unit_price REAL NOT NULL,
  PRIMARY KEY (order_id, product_id),
  FOREIGN KEY (order_id) REFERENCES orders(order_id),
  FOREIGN KEY (product_id) REFERENCES products(product_id)
);
"""}
]

# -------- Variant C: Rich DDL + inserts --------
context_C = [
  {'role':'system','content': """You are a Text-to-SQL assistant. 
- Output ONLY the final SQL for SQLite.
- Prefer explicit JOIN ... ON ...
- Use GROUP BY when aggregating.
- Do NOT invent columns or tables.
- If the question is impossible, answer exactly: 'I cannot answer that with the given schema.'"""},
  {'role':'system','content': """
-- ECommerce schema (SQLite)
CREATE TABLE customers (
  customer_id INTEGER PRIMARY KEY,
  name TEXT NOT NULL,
  email TEXT UNIQUE,
  city TEXT
);
CREATE TABLE products (
  product_id INTEGER PRIMARY KEY,
  name TEXT NOT NULL,
  category TEXT,
  price REAL NOT NULL
);
CREATE TABLE orders (
  order_id INTEGER PRIMARY KEY,
  customer_id INTEGER NOT NULL,
  order_date TEXT NOT NULL,
  status TEXT,
  FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
CREATE TABLE order_items (
  order_id INTEGER NOT NULL,
  product_id INTEGER NOT NULL,
  quantity INTEGER NOT NULL,
  unit_price REAL NOT NULL,
  PRIMARY KEY (order_id, product_id),
  FOREIGN KEY (order_id) REFERENCES orders(order_id),
  FOREIGN KEY (product_id) REFERENCES products(product_id)
);

-- A few sample rows
INSERT INTO customers (customer_id, name, email, city) VALUES
  (1, 'Alice Johnson', 'alice@example.com', 'Madrid'),
  (2, 'Bob Smith', 'bob@example.com', 'Barcelona'),
  (3, 'Carla Ruiz', 'carla@example.com', 'Valencia');

INSERT INTO products (product_id, name, category, price) VALUES
  (10, 'Wireless Mouse', 'Electronics', 25.99),
  (11, 'Mechanical Keyboard', 'Electronics', 89.50),
  (12, 'Notebook A5', 'Stationery', 4.20),
  (13, 'USB-C Cable', 'Electronics', 8.99);

INSERT INTO orders (order_id, customer_id, order_date, status) VALUES
  (100, 1, '2024-07-15', 'shipped'),
  (101, 1, '2024-07-20', 'delivered'),
  (102, 2, '2024-08-01', 'processing'),
  (103, 3, '2024-08-02', 'cancelled');

INSERT INTO order_items (order_id, product_id, quantity, unit_price) VALUES
  (100, 10, 1, 25.99),
  (100, 12, 3, 4.20),
  (101, 11, 1, 89.50),
  (101, 13, 2, 8.99),
  (102, 10, 2, 24.50),
  (103, 12, 5, 4.00);
"""}
]

3. Question to test

In [30]:
questions = [
  "List name and email of customers located in Madrid.",
  "Get order_id of orders placed by Alice Johnson in July 2024.",
  "For each customer, show total amount spent (sum of quantity*unit_price).",
  "Top 3 best-selling products by total quantity sold.",
  "Which products have never been ordered?",
  "Average order value per month in 2024 (exclude cancelled).",
  "Show order_id and total amount for all delivered orders.",
  "Find customers who placed no orders."
]

4. Runner (output)

In [32]:
from copy import deepcopy

contexts = {
  "A_minimal": context_A,
  "B_DDL_only": context_B,
  "C_DDL_plus_rows": context_C,
}

def run_all():
    results = []
    for cname, ctx in contexts.items():
        for q in questions:
            try:
                sql = return_CCRMSQL(q, deepcopy(ctx))  # uses your lab function
            except NameError:
                sql = "[ERROR] return_CCRMSQL not defined in this notebook."
            print(f"\n### {cname} :: {q}\n{sql}\n" + "-"*80)
            results.append((cname, q, sql))
    return results

results = run_all()


### A_minimal :: List name and email of customers located in Madrid.
```sql
SELECT name, email
FROM customers
WHERE city = 'Madrid';
```
--------------------------------------------------------------------------------

### A_minimal :: Get order_id of orders placed by Alice Johnson in July 2024.
```sql
SELECT order_id
FROM orders
WHERE customer_id = (SELECT customer_id FROM customers WHERE name = 'Alice Johnson')
AND order_date BETWEEN '2024-07-01' AND '2024-07-31';
```
--------------------------------------------------------------------------------

### A_minimal :: For each customer, show total amount spent (sum of quantity*unit_price).
```sql
SELECT c.customer_id, c.name, SUM(oi.quantity * oi.unit_price) AS total_amount_spent
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.name;
```
--------------------------------------------------------------------------------

### A_minimal :: Top 3 best-se

# One‑Page Report — SQL Prompt Variants

**Goal.** Compare multiple text-to-sql prompt styles for an e-commerce schema and observe quality. 

## Variants Used
1. **A_minimal** : table/columns names + FK list only. This way we give GPT a bare minimum schema, so that we can study how prompt richness changes SQL quality. 
2. **B_DDL_only**: Full create table DLL, no sample rows. In here we only CREATE table statements. 
3. **C_DD_plus_rows** : DDL + a few INSERT rows + explicit assistant rules.

## Findings
1) **C_DDL_plus_rows** produced the most reliable SQL overall (fewer hallucinations; correct joins/aggregations).
2) **B_DDL_only** was good for joins, but sometimes missed semantics (e.g., forgetting to exclude `status='cancelled'`).
3) **A_minimal** struggled: invented columns, wrong joins, GROUP BY issues.

## Failure modes
- Hallucinated columns: `orders.total_price`- from the example : “Show order_id and total amount for all delivered orders.”. "orders" table does not have a "total_price" column. Total must be computed from "order_items.quantity. 
- Anti-join mistakes: products never ordered → returned all products.
- Date filters: inclusive BETWEEN mistakes.

## Learnings
- Representation matters: DDL + sample rows >> plain column names.
- Few-shot helps: show join, aggregation, anti-join once..
- Dialect clarity avoids drift.



<!-- 4. Runner (output) -->